In [224]:
import requests
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate 
from langchain_core.tools import tool, StructuredTool, InjectedToolArg
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Optional, List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
import yfinance as yf
import pandas_ta as ta
import json

load_dotenv()

True

In [225]:
NEWS_DATA_API=os.getenv('NEWS_DATA_API')

In [226]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a query parameter builder for the NewsData.io API focusing strictly on Indian stock market catalysts (NSE/BSE).

Target Topic: {topic}

Step 1: Choose your search scope (CHOOSE EXACTLY ONE):
- 'q=...' (Broad search across the whole article)
- 'qInTitle=...' (Strict search only in the headline)
- 'qInMeta=...' (Search titles, URLs, and meta tags)
- 'timeframe=...' (Search by timeframe)
- 'category=...' (Search by category)

CRITICAL: Never combine these. Pick only one.

Step 2: Apply these mandatory rules:
1. Use actionable single-stock catalyst terms (e.g., "quarterly results", "order win", "acquisition").
2. Strictly DO NOT include broad market phrases like "Sensex" or "Nifty".
3. Always append '&category=business&country=in'.

Validation Rules:
- STRICTLY EXCLUDE foreign equitie  s (e.g., Nike, Tesla, Apple, Google). Only include companies actively listed on the NSE or BSE.
- STRICTLY EXCLUDE general editorial comparisons or educational articles without fresh catalysts.
- For each valid event, state the exact NSE/BSE ticker symbol (e.g., UNIONBANK, INFY, WIPRO).

Output Format:
- Output ONLY the final query string separated by '&'.
- Do NOT include the base URL or conversational explanations.
- Example output: qInTitle=order+win+OR+acquisition&category=business&country=in
""",
    input_variables=["topic"]
)

In [227]:
model = ChatOllama(model="gemma4:e4b", temperature=0.7, num_ctx=8192)

In [228]:
parser = StrOutputParser()

In [229]:
class Article(BaseModel):
    title: str = Field(..., description="The title of the news article")
    description: Optional[str] = Field(..., description="A brief description of the news article")
    source: Optional[str] = Field(..., description="The source of the news article")

In [230]:
class ArticleList(BaseModel):
    articles: list[Article] = Field(description="List of 5 news articles")

In [231]:
articles_parser = PydanticOutputParser(pydantic_object=ArticleList)


In [232]:
@tool
def generate_query_params(topic: str):
    """Generate a formatted URL query parameter string for fetching news articles."""
    # print("generate_query_params called with topic:", topic)
    model = ChatOllama(model="gemma4:e4b", temperature=0.7, max_tokens=4000)
    chain = prompt | model | parser

    prompt_text = prompt.invoke({'topic': topic})
    print(prompt_text)

    query_params = chain.invoke({'topic' : topic})
    # print(query_params)

    return query_params

In [ ]:
class NewsQueryParams(BaseModel):
    topic : str = Field(..., description="The topic to search news articles for")

# @tool
def fetch_news_articles(topic: str):
    """ Tool to fetch news articles. MUST only be called after generating query parameters. """

    query_chain = prompt | model | parser
    query_params = query_chain.invoke({"topic": topic})

    print("query_params_before_fetching: ",query_params)
    
    url = f"https://newsdata.io/api/1/latest?apikey={NEWS_DATA_API}&{query_params}&size=25"
    response = requests.get(url)
    data = response.json()

    # 1. Check for API success
    if data.get("status") == "success":
        clean_articles = []
        
        # 2. Extract only the essential text needed by the financial analyst prompt
        for article in data.get("results", []):
            clean_articles.append({
                "title": article.get("title"),
                "description": article.get("description"),
                "source": article.get("source_name")
            })
            
        # 3. Return the minimized list of dictionaries, NOT the raw JSON
        return clean_articles

    return data

fetch_news_articles_tool = StructuredTool.from_function(
    func=fetch_news_articles,
    name="fetch_news_articles",
    description="Fetch relevant news articles by passing relevant query parameters to the API",
    args_schema=NewsQueryParams
)

In [ ]:
agent = create_agent(
    model=model,
    tools=[fetch_news_articles_tool],
)

format_instructions = articles_parser.get_format_instructions()

prompt_text = f"""You are an equity research analyst covering individual stocks on the NSE and BSE.

Step 1:
Call `fetch_news_articles` with a query focused on company-specific corporate actions (such as quarterly earnings surprises, mega contract wins, mergers, or regulatory penalties).

Step 2:
From the fetched results, filter out and ignore:
- Broad market index wrap-ups (e.g., Sensex/Nifty up/down commentary).
- Trading holiday calendars and exchange scheduling notices.
- Global macroeconomic recaps (crude oil prices, US bond yields).

Step 3:
Extract UP TO 5 real Indian companies present in the fetched text. 
CRITICAL RULE:
# - NEVER invent, synthesize, or use placeholder companies (e.g., do NOT generate 'TechCorp', 'SteelGiant', 'BioPharma').
- If only 1 or 2 real Indian companies exist in the fetched results, return ONLY those 1 or 2.
- If no real companies are found, return an empty list.

Output strictly according to these formatting instructions:
{format_instructions}"""

response = agent.invoke({"messages": [HumanMessage(content=prompt_text)]})
raw_content = response["messages"][-1].content
# print("Raw content from agent:\n", raw_content)
structured_result = articles_parser.parse(raw_content)

for i, item in enumerate(structured_result.articles):
    print(f"Article {i+1}:\nTitle: {item.title}\nDescription: {item.description}\nSource: {item.source}\n")

query_params_before_fetching:  qInTitle=quarterly+results+OR+order+win+OR+acquisition+OR+earnings+OR+dividend+OR+buyback&category=business&country=in


In [235]:
# print(clean_json_str)
# article_list = ArticleList.model_validate_json(clean_json_str)
# print(article_list)

In [236]:
import pandas as pd
import requests
import io

def get_valid_nse_tickers():
    """
    Downloads the official active equity master list directly from the NSE 
    and returns a set of valid ticker symbols.
    """
    url = "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv"
    
    # NSE blocks standard python user agents, so we spoof a standard browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Read the CSV into pandas
        df = pd.read_csv(io.StringIO(response.text))
        
        # The 'SYMBOL' column contains the official tickers (e.g., 'RELIANCE', 'TCS')
        valid_symbols = set(df['SYMBOL'].str.strip().str.upper())
        print(f"Loaded {len(valid_symbols)} active NSE tickers.")
        return valid_symbols
        
    except Exception as e:
        print(f"Failed to fetch NSE master list: {e}")
        # Fallback to a small critical list if the download fails
        return {"RELIANCE", "TCS", "HDFCBANK", "INFY", "ITC"} 

# Run this once globally
VALID_NSE_TICKERS = get_valid_nse_tickers()

Loaded 2593 active NSE tickers.


In [237]:
# %%
class TickerTechnicalScore(BaseModel):
    ticker: str = Field(description="The official NSE ticker symbol, or 'NONE'")
    open: float | None = Field(default=None, description="The opening price")
    close: float | None = Field(default=None, description="The closing price")
    high: float | None = Field(default=None, description="The highest price")
    low: float | None = Field(default=None, description="The lowest price")
    rsi14: float | None = Field(default=None, description="14-day RSI")
    macd12_26_9: float | None = Field(default=None, description="MACD value")
    ema20: float | None = Field(default=None, description="20-day EMA")
    news_title: str = Field(description="Title of the catalyst article")
    descriptioin: str = Field(description="Description of article")
    conclusion: str = Field(description="Actionable takeaway on stock movement")

class TickersTechnicalReport(BaseModel):
    reports: list[TickerTechnicalScore] = Field(description="List of technical evaluations for the identified stocks")

In [238]:
@tool
def get_ticker_technicals(ticker: str):
    """Fetches technical indicators (RSI, MACD, EMA) for a given NSE ticker symbol."""
    
    # Clean up what the LLM gave us
    base_ticker = ticker.strip().upper().replace(".NS", "").replace(".BO", "")
    
    # 1. GROUND TRUTH CHECK: Is this a real NSE company?
    if base_ticker not in VALID_NSE_TICKERS:
        return {
            "error": f"'{base_ticker}' is not an active NSE listed equity. Skipping analysis."
        }
    
    # 2. Re-append the .NS suffix for Yahoo Finance
    ticker_clean = f"{base_ticker}.NS"

    try:
        df = yf.Ticker(ticker_clean).history(period="6mo")
        
        # 3. Check if yfinance actually returned data
        if df.empty or len(df) < 30:
            return {"error": f"Insufficient trading data found for {ticker_clean}."}

        # Calculate technical indicators safely
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)
        df.ta.ema(length=20, append=True)

        return {
            "latest_open": round(float(df["Open"].iloc[-1]), 2),
            "latest_close": round(float(df["Close"].iloc[-1]), 2),
            "latest_high": round(float(df["High"].iloc[-1]), 2),
            "latest_low": round(float(df["Low"].iloc[-1]), 2),
            "latest_rsi": round(float(df["RSI_14"].iloc[-1]), 2),
            "latest_macd": round(float(df["MACD_12_26_9"].iloc[-1]), 2),
            "latest_ema20": round(float(df["EMA_20"].iloc[-1]), 2)
        }
    except Exception as e:
        return {"error": f"Failed calculating technicals for {ticker_clean}: {str(e)}"}

In [239]:
agent_technical = create_agent(
    model=model,
    tools=[get_ticker_technicals]
)


ticker_technical_parser = PydanticOutputParser(pydantic_object=TickersTechnicalReport)

articles_text = "\n".join([
    f"- {art.title} (Source: {art.source}): {art.description}"
    for art in structured_result.articles
])

prompt_content = f"""
You are a technical market analyst.
Review the following news articles:

{articles_text}

Task:
Process EVERY single article provided above into your final output. Do not skip any.

For each article, follow this logic:
1. Identify if it explicitly mentions a specific Indian company listed on the NSE or BSE.
2. If YES (Valid Indian Ticker):
   - Extract the official NSE ticker symbol (e.g., RELIANCE, TCS, APLAPOLLO).
   - Call the `get_ticker_technicals` tool for this ticker.
   - Write a conclusion evaluating both the news catalyst and the technical data.
3. If NO (Foreign company, macroeconomic event, or unnamed entity):
   - Set the ticker strictly to the word: NONE
   - DO NOT call the `get_ticker_technicals` tool.
   - Leave the technical fields (open, close, rsi, macd, etc.) as null.
   - Write a conclusion summarizing the news impact on the broader market or sector.

Return your final answer strictly following these formatting instructions:
{ticker_technical_parser.get_format_instructions()}
"""
agent_response = agent_technical.invoke({
    "messages": [HumanMessage(content=prompt_content)]
})

raw_text = agent_response["messages"][-1].content

# Use the Pydantic parser directly to handle clean deserialization
try:
    parsed_report = ticker_technical_parser.parse(raw_text)
    for item in parsed_report.reports:
        print(f"Ticker: {item.ticker}")
        print(f"RSI: {item.rsi14} | Close: ₹{item.close}")
        print(f"Conclusion: {item.conclusion}\n" + "-"*40)
except Exception:
    # Fallback to manual JSON parsing if the local LLM dropped part of the Pydantic format
    clean_json = raw_text.replace("```json", "").replace("```", "").strip()
    data = json.loads(clean_json)
    reports = data.get("reports", data) if isinstance(data, dict) else data
    for item in reports:
        print(f"Ticker: {item.get('ticker')} | Conclusion: {item.get('conclusion')}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [241]:
print(raw_text)

Please provide the news articles you would like me to review. I am ready to act as the technical market analyst and process each article according to your specified logic and output the final result in the required JSON format.


In [ ]:
class FinancialTerm(BaseModel):
    term: str = Field(description="The financial/trading term")
    definition: str = Field(description="Plain English definition")
    applied_to_news: str = Field(description="How this concept applies directly to this specific news event")

class CatalystDeepDive(BaseModel):
    ticker: str = Field(description="The NSE ticker or NONE")
    catalyst_summary: str = Field(description="What happened and whether it beat/missed market expectations")
    company_impact: str = Field(description="Impacts on revenue, margins, balance sheet, or operational efficiency")
    competitors_impact: str = Field(description="Effect on listed peers, market share battles, or supplier pricing power")
    macro_ripple: str = Field(description="Broader implications for Indian economy, interest rates, or raw material demand")
    key_terms_taught: List[FinancialTerm] = Field(description="1-2 professional financial/economic concepts to learn from this event")
    # Updated to focus strictly on fundamentals
    trader_takeaway: str = Field(description="Actionable trading setup or market sentiment shift based purely on this fundamental catalyst")

    
class MarketIntelligenceReport(BaseModel):
    analyses: List[CatalystDeepDive]

In [ ]:
market_mentor_parser = PydanticOutputParser(pydantic_object=MarketIntelligenceReport)
market_mentor_format_instructions = market_mentor_parser.get_format_instructions()

In [ ]:
# You are a senior hedge fund portfolio manager and trading mentor.
# Your job is to analyze corporate actions and news catalysts, break down economic causality, and teach market mechanics.
# - Source: {source}

mentor_prompt = PromptTemplate(
    template="""You are a senior hedge fund portfolio manager and mentor.
    Analyze the following catalyst, then populate the required JSON fields.

    News Catalyst:
    - Title: {title}
    - Source: {source}
    - Details: {description}

    Market Info:
    - Ticker: {ticker}
    - Close Price: {close}
    - RSI (14-day): {rsi}
    - 20-Day EMA: {ema20}
    - MACD: {macd}

    Analyze the event across five dimensions:
    1. FIRST-ORDER EFFECT (The Business): Revenue vs margin impact, pricing power, or capacity.
    2. SECOND-ORDER EFFECT (Competitors & Read-Across): Peer impact and market share dynamics.
    3. MACRO & SUPPLY CHAIN: Broader demand, capex, or raw material trends.
    4. ECONOMIC CONCEPT TO MASTER: Explain 1-2 core institutional concepts.
    5. TRADER PLAYBOOK: Synthesize the news with the provided technicals. If technicals are marked 'N/A', ignore them and provide a purely fundamental takeaway.
    """,
    input_variables=["title", "source", "description", "ticker", "close", "rsi", "ema20", "macd"]
)

In [240]:
parsed_report = ticker_technical_parser.parse(raw_text)
print(parsed_report)

OutputParserException: Invalid json output: Please provide the news articles you would like me to review. I am ready to act as the technical market analyst and process each article according to your specified logic and output the final result in the required JSON format.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [ ]:
model = ChatOllama(model="gemma4:e4b", temperature=0.3, num_ctx=8192)

structured_mentor_model = model.with_structured_output(CatalystDeepDive)
mentor_chain = mentor_prompt | structured_mentor_model

In [ ]:
final_mentor_analyses = []
print("Generating Hedge Fund Deep Dives...\n")

for item in parsed_report.reports:
    title = getattr(item, 'news_title', 'Unknown Title')
    desc = getattr(item, 'descriptioin', getattr(item, 'description', 'No description provided'))
    source = getattr(item, 'source', 'Unknown Source')

    if title == "Unknown Title" or not title.strip():
        continue

    print(f"Analyzing catalyst for: {item.ticker}...")
    
    try:
        # We safely pass the technicals, replacing missing values with "N/A"
        analysis = mentor_chain.invoke({
            "title": title,
            "source": source,
            "description": desc,
            "ticker": item.ticker,
            "close": f"₹{item.close}" if getattr(item, 'close', None) is not None else "N/A",
            "rsi": str(round(item.rsi14, 2)) if getattr(item, 'rsi14', None) is not None else "N/A",
            "ema20": str(round(item.ema20, 2)) if getattr(item, 'ema20', None) is not None else "N/A",
            "macd": str(round(item.macd12_26_9, 2)) if getattr(item, 'macd12_26_9', None) is not None else "N/A"
        })
        
        final_mentor_analyses.append(analysis)
        print("  ✓ Success")
        
    except Exception as e:
        print(f"  ✗ Error analyzing {item.ticker}: {e}")

Generating Hedge Fund Deep Dives...



In [ ]:
for i, report in enumerate(final_mentor_analyses):
    print(f"\n\nReport {i + 1}")
    print("\ncatalyst_summary: " , report.catalyst_summary)
    print("\ncompetitors_impact: " , report.competitors_impact)
    print("\nmacro_ripple: " , report.macro_ripple)

    print("\nlearnings: ")
    for key_term in report.key_terms_taught:
        print(f"\nTerm: {key_term.term} \n Definition: {key_term.definition}")

    print("\ntrader_takeaway: " , report.trader_takeaway)
